# Informe de Experimentación — MLP Clasificador de Imágenes

## Experimento 1 — Modelo Base corregido (sin Data Leakage)
* **Configuración:** Adam, lr=0.001, batch=32, 10 épocas.
* **Métricas:** Train: 57.2% / Val: 51.5% / Val Loss: 1.3534.
* **Diagnóstico:** Después de corregir el data leakage, el modelo siguió generalizando bastante bien y no mostró overfitting fuerte. Para ser la primera corrida con una red simple y 9 clases, los resultados fueron bastante buenos y dejaron margen claro para seguir mejorando con más épocas o una arquitectura más grande.

## Experimento 2 — Más épocas
* **Configuración:** Adam, lr=0.001, batch=32, 30 épocas.
* **Métricas:** Train: 74.5% / Val: 52.7% / Val Loss: 1.6100.
* **Diagnóstico:** Al aumentar la cantidad de épocas, el modelo siguió mejorando considerablemente sobre train pero casi no mejoró en validation. Además, la validation loss aumentó, indicando el comienzo de overfitting. Esto sugirió que el modelo empezó a memorizar el conjunto de entrenamiento sin lograr una mejor generalización. Aun así, se decidió mantener las 30 épocas para los próximos experimentos con el objetivo de evaluar técnicas de regularización y data augmentation. Probablemente, para un modelo final convenga reducir la cantidad de épocas o aplicar early stopping para evitar el sobreajuste.

## Experimento 3 — Oversampling manual de clases minoritarias
* **Configuración:** Adam, lr=0.001, batch=32, 30 épocas. Balanceo en TRAIN duplicando las muestras de las clases "7" y "8".
* **Métricas:** Train: 77.43% / Val: 55.03% / Val Loss: 1.4131.
* **Diagnóstico:** Al balancear las clases el modelo mejoró la generalización, subiendo el accuracy en validación y frenando la pérdida (comparado con el Experimento 2). Sin embargo, al duplicar imágenes idénticas, el modelo memorizó de más. Esto se ve en el salto del train accuracy, manteniendo el overfitting. El balanceo sirve, pero para el próximo experimento hay que rellenar con data augmentation controlado (sumando las ~23 y ~12 fotos) para que no memorice repetidos.

## Experimento 4 — Oversampling controlado con Data Augmentation
* **Configuración:** Adam, lr=0.001, batch=32, 30 épocas. Balanceo en TRAIN sumando muestras aleatorias exactas (~23 a la clase "8" y ~12 a la clase "7") combinadas con Albumentations.
* **Métricas:** Train: 77.96% / Val: 61.53% / Val Loss: 1.6462.
* **Diagnóstico:** El balanceo controlado con aumentación de datos dio el mejor resultado en validación hasta ahora.

## Experimento 4 — Limpieza de foto negra

* **Configuración:** Adam, lr=0.001, batch=32, 30 épocas.
* **Cambio:** Se eliminó `aug_0_F2.large.jpg` del dataset antes del split — era una imagen completamente negra que no aportaba información visual real.
* **Métricas:** Train: 79.63% / Val: 61.54% / Val Loss: 1.5371.
* **Diagnóstico:** Resultados casi idénticos al experimento anterior, lo cual es esperable — una foto menos no cambia drásticamente el entrenamiento. La val loss bajó levemente (1.5371 vs 1.6462), lo que sugiere que el modelo generaliza un poco mejor sin la imagen corrupta.


## Experimento 5 — Early Stopping

* **Configuración:** Adam, lr=0.001, batch=32, máximo 50 épocas con early stopping (patience=5).
* **Cambio:** Se implementó early stopping — el entrenamiento se detiene automáticamente si la val loss no mejora durante 5 épocas consecutivas, y se guarda el mejor modelo.
* **Métricas:** Train: 68.89% / Val: 55.03% / Val Loss: 1.3852. Early stopping en época 17.
* **Diagnóstico:** El ES funcionó correctamente y frenó el overfitting — la brecha train/val bajó a ~14 puntos respecto a los ~18 del experimento anterior. El modelo guardado corresponde al mejor checkpoint durante el entrenamiento, no a la última época.

## Experimento 6 — Augmentations avanzadas
* **Configuración:** Adam, lr=0.001, batch=32, early stopping (patience=5). Se agregaron augmentations clínicas: HFlip, VFlip, RandomRotate90, BrightnessContrast, CLAHE, GaussNoise, CoarseDropout, ElasticTransform, HueSaturationValue.
* **Métricas:** Train: 57.59% / Val: 54.44% / Val Loss: 1.3635.
* **Diagnóstico:** Las augmentations bajaron el train accuracy (más difícil de memorizar) pero no mejoraron la validación respecto al experimento 6. Una red sin regularización ni BatchNorm no puede aprovechar bien las augmentations. Próximo paso: ajustar la arquitectura.

## Experimento 8 — Ajuste de arquitectura (red más grande)
* **Configuración:** Adam, lr=0.001, batch=32, early stopping (patience=5). Arquitectura ampliada a 512→256.
* **Métricas:** Train: 63.52% / Val: 53.25% / Val Loss: 1.4261. Early stopping en ~4.5 minutos.
* **Diagnóstico:** Agrandar la red subió el train accuracy (63% vs 57%) pero la validación bajó levemente (53% vs 54%). La brecha train/val se mantiene similar — el modelo tiene más capacidad pero sigue sin poder generalizarla. Próximo paso: regularización (Weight Decay, Dropout).

## Experimento 9 — Más capas (512→256→128)
* **Configuración:** Adam, lr=0.001, batch=32, early stopping (patience=5). Arquitectura con tres capas ocultas: 512→256→128.
* **Métricas:** Train: 58.70% / Val: 53.25% / Val Loss: 1.2900.
* **Diagnóstico:** Agregar una capa extra empeoró el train (58% vs 63% del experimento anterior) y no mejoró la validación. Con más capas y sin regularización el gradiente se pierde, haciendo más difícil el entrenamiento. Se descartó esta arquitectura y se volvió a 512→256 para continuar con técnicas de regularización.

### Experimento 10 — Retorno a Arquitectura Base + Regularización L2 (Weight Decay)
* Configuración: Adam, lr=0.001, batch=32, max_epochs=50 con Early Stopping (patience=5). Arquitectura base sin capas extras + weight_decay=0.0001.
* Métricas: Train: 61.66% / Val: 46.74% / Val Loss: 1.4493. Early stopping en época 22.
* Diagnóstico: Al quitar la tercera capa oculta y agregar Weight Decay (1e-4), el modelo mostró una dinámica mucho más estable en las primeras épocas, llegando a tocar picos de 60% de accuracy en validación cerca de la época 20. Sin embargo, la penalización resultó muy leve para contener el sobreajuste al final; el modelo terminó memorizando el ruido de train, lo que disparó la pérdida de validación y tiró abajo el accuracy, activando el Early Stopping.


### Experimento 11 — Regularización L2 Más Fuerte (Weight Decay = 1e-3)
* Configuración: Adam, lr=0.001, batch=32, max_epochs=50 con Early Stopping (patience=5). Arquitectura base sin capas extras + weight_decay=0.001.
* Métricas: Train: 57.77% / Val: 55.02% / Val Loss: 1.2604. Early stopping en época 19.
* Diagnóstico: Subir el Weight Decay a 1e-3 estabilizó por completo la convergencia del modelo. Al penalizar con mayor fuerza los pesos grandes, se logró suavizar el espacio de optimización, eliminando la caída libre en validación que sufría el experimento anterior pasadas las 20 épocas. La pérdida de validación bajó de forma controlada hasta un piso de 1.2604 y la brecha train/val se redujo a solo 2.7 puntos, demostrando que una regularización matemática más agresiva frena eficazmente la memorización de ruido.

#dropout

#batch norm



## Configuración Final Ganadora

| Parámetro | Valor |
|---|---|
| Arquitectura | 768 → 192 |
| Optimizer | Adam |
| Learning Rate | 0.0001 |
| Weight Decay | 1e-3 |
| Batch Size | 32 |
| Épocas | 20 |
| Dropout | 0.25 (capa 1) / 0.0 (capa 2) |
| BatchNorm | Sí |
| Resolución | 64×64 |
| Augmentations | HFlip, VFlip, Rotate90, BrightnessContrast(0.4), CLAHE(0.3), CoarseDropout |



Cerramos esta etapa y pasamos al **Archivo 2** para seguir probando.

